In [2]:
import numpy as np
import joblib
import librosa
import os

In [4]:
def extract_all_features(file_path, label="unknown", sr=22050, segment_duration=3):
    """
    Trích xuất toàn bộ feature từ file audio theo cấu trúc features_3_sec.csv
    
    Parameters:
        file_path (str): Đường dẫn file audio
        label (str): Nhãn (thể loại hoặc tên)
        sr (int): Sampling rate
        segment_duration (float): Thời lượng mỗi đoạn tính feature (giây)
        
    Returns:
        features_list (list): Danh sách các dòng feature
    """
    # Load file
    y, sr = librosa.load(file_path, sr=sr)
    total_duration = librosa.get_duration(y=y, sr=sr)
    
    features_list = []
    num_segments = int(total_duration // segment_duration)
    
    for i in range(num_segments):
        start_sample = int(i * segment_duration * sr)
        end_sample = int(start_sample + segment_duration * sr)
        segment = y[start_sample:end_sample]
        
        if len(segment) < segment_duration * sr:
            continue
        
        # Spectral & chroma
        chroma_stft = librosa.feature.chroma_stft(y=segment, sr=sr)
        rms = librosa.feature.rms(y=segment)
        spec_cent = librosa.feature.spectral_centroid(y=segment, sr=sr)
        spec_bw = librosa.feature.spectral_bandwidth(y=segment, sr=sr)
        rolloff = librosa.feature.spectral_rolloff(y=segment, sr=sr)
        zcr = librosa.feature.zero_crossing_rate(segment)
        
        # Harmonic & percussive
        harmony, perceptr = librosa.effects.hpss(segment)
        
        # Tempo
        tempo = librosa.beat.tempo(y=segment, sr=sr)[0]
        
        # MFCC
        mfcc = librosa.feature.mfcc(y=segment, sr=sr, n_mfcc=20)
        
        # Build row
        row = [
            os.path.basename(file_path),  # filename
            total_duration,               # length
            np.mean(chroma_stft), np.var(chroma_stft),
            np.mean(rms), np.var(rms),
            np.mean(spec_cent), np.var(spec_cent),
            np.mean(spec_bw), np.var(spec_bw),
            np.mean(rolloff), np.var(rolloff),
            np.mean(zcr), np.var(zcr),
            np.mean(harmony), np.var(harmony),
            np.mean(perceptr), np.var(perceptr),
            tempo
        ]
        
        # Thêm MFCC mean/var
        for m in mfcc:
            row.append(np.mean(m))
            row.append(np.var(m))
        
        # Label
        row.append(label)
        
        features_list.append(row)
    
    return features_list


In [5]:
# Đọc file audio data
audio_path = "Assets/Input_sample/3uvxf7uvy5.mp3" # Thay đổi đường dẫn tới file audio

In [6]:
features = extract_all_features(audio_path)

/var/folders/2f/t67gd14j5gg9gy71bywcdqpm0000gn/T/ipykernel_81654/522749949.py:41: FutureWarning: librosa.beat.tempo
	This function was moved to 'librosa.feature.rhythm.tempo' in librosa version 0.10.0.
	This alias will be removed in librosa version 1.0.
  tempo = librosa.beat.tempo(y=segment, sr=sr)[0]


In [7]:
min_max_scaler = joblib.load("DL/min_max_scaler.save")
standard_scaler = joblib.load("DL/standard_scaler.save")

/opt/miniconda3/envs/py310-env/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.7.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/miniconda3/envs/py310-env/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.7.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [8]:
features = [row[1:-1] for row in features]
features_scaled = min_max_scaler.transform(features)

/opt/miniconda3/envs/py310-env/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [9]:
features_scaled = [row[1:] for row in features_scaled]
features_scaled = standard_scaler.transform(features_scaled)

/opt/miniconda3/envs/py310-env/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [1]:
import numpy as np
import tensorflow as tf
import keras as k


2025-08-15 23:02:07.142893: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [10]:
index_label = {0: 'blues', 1: 'classical', 2: 'country', 3: 'disco', 4: 'hiphop', 5: 'jazz', 6: 'metal', 7: 'pop', 8: 'reggae', 9: 'rock'}


In [11]:
model = k.models.Sequential([
    k.layers.Dense(1024, activation='relu', input_shape=(57,)),
    k.layers.Dropout(0.3),

    k.layers.Dense(512, activation='relu'),
    k.layers.Dropout(0.3),

    k.layers.Dense(256, activation='relu'),
    k.layers.Dropout(0.3),

    k.layers.Dense(128, activation='relu'),
    k.layers.Dropout(0.3),

    k.layers.Dense(64, activation='relu'),
    k.layers.Dropout(0.3),

    k.layers.Dense(10, activation='softmax'),
])

/opt/miniconda3/envs/py310-env/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [13]:
model.load_weights('DL/model.weights.h5')

In [14]:
pred_proba = model.predict(features_scaled)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step


In [15]:
pred_class = np.argmax(pred_proba, axis=1)

In [16]:
print(index_label[pred_class[0]])

disco
